In [1]:
# Install required libraries for RAG pipeline
!pip install langchain langchain-community langchain-core
!pip install chromadb
!pip install sentence-transformers
!pip install pypdf2
!pip install google-generativeai
!pip install tiktoken
!pip install langchain-text-splitters
!pip install requests
!pip install google-genai
!pip install -U `langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [3]:
import sys
!{sys.executable} -m pip install -U 'langchain-huggingface'

# Importing all the libraries required
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import SentenceTransformersTokenTextSplitter
# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import GooglePalm
from langchain_classic.chains import RetrievalQA
import chromadb
import requests
import os
import tempfile

/tmp/ipykernel_13466/4099384689.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [4]:
# Extraction of text from PDF file
def extract_pdf_content(pdf_path):
  text = ""
  # Download the PDF if the path is a URL
  if pdf_path.startswith("http://") or pdf_path.startswith("https://"):
    response = requests.get(pdf_path)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    # Create a temporary file to save the PDF content
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_pdf:
      temp_pdf.write(response.content)
      temp_pdf_path = temp_pdf.name

    try:
      with open(temp_pdf_path, "rb") as pdf_file:
        pdf_reader = PdfReader(pdf_file)
        for page in pdf_reader.pages:
          text += page.extract_text()
    finally:
      os.remove(temp_pdf_path) # Clean up the temporary file
  else: # Assume it's a local file path
    with open(pdf_path, "rb") as pdf_file:
      pdf_reader = PdfReader(pdf_file)
      for page in pdf_reader.pages:
        text += page.extract_text()
  return text

In [5]:
# Setting the parameters for text splitting
sent_text_splitter = SentenceTransformersTokenTextSplitter(
    chunk_overlap=10, # Overlap tokens for context continuity
    model_name='sentence-transformers/all-MiniLM-L6-v2', # Embedding model for tokenization
    tokens_per_chunk=100 # Chunk size (tunable for your use case)
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# Chunking the text into smaller pieces for better processing and retrieval
def chunk_text(text,file_name):
  chunks = []
  for chunk in sent_text_splitter.split_text(text):
    chunks.append({"content":chunk,
                   "metadata":{"filename":file_name}})
  return chunks

In [7]:
# Initializing the embedding model for converting text chunks into vector representations
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# Storing the chunks in ChromaDB for efficient retrieval based on semantic similarity
def store_in_chroma(chunks,persist_directory="./chroma_store"):
  texts = [c["content"] for c in chunks]
  metadatas = [c["metadata"] for c in chunks]
  db = Chroma.from_texts(texts, embedding_model,metadatas=metadatas, persist_directory=persist_directory)
  return db

In [9]:
pdf_path = "https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf"
filename = pdf_path.split("/")[-1]
text = extract_pdf_content(pdf_path)
chunks = chunk_text(text,filename)
db = store_in_chroma(chunks)

In [10]:
# Basic retrieval function to get relevant chunks from ChromaDB based on query similarity
def search_chroma(query,db,top_k=5):
  results = db.similarity_search(query,k=top_k)
  chunks = [{"content":d.page_content,"metadata":d.metadata} for d in results]
  return chunks

In [12]:
# --- Configure Gemini API Key ---
# Securely load your Google Gemini API key from Colab userdata.
# Use Case: Keeps credentials safe and enables authenticated LLM access.
import google.generativeai as genai # Correct import for genai.configure
from google.colab import userdata
api_key = userdata.get('G_API')
genai.configure(api_key=api_key)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [13]:
# Context retriever function to retrieve relevant context from ChromaDB for a given query.
def retrieve_context(query, db, top_k=5):

    chunks = search_chroma(query, db, top_k)

    return "\n\n".join(
        [c["content"] for c in chunks]
    )

In [19]:
# Generation pipeline for generating answers based on the retrieved context.
def agentic_rag_answer(query, db):

    model = genai.GenerativeModel(
        "models/gemini-2.5-flash",
        generation_config={
            "temperature": 0.7
        }
    )
    memory = []

    current_query = query

    for step in range(3):

        context = retrieve_context(
            current_query,
            db
        )

        memory.append(context)

        planning_prompt = f"""
        User Question: {query}

        Retrieved Context:
        {context}

        Based on the User Question and Retrieved Context, decide if you have enough information to answer the User Question fully.
        If not, suggest a concise NEXT_QUERY to search for more information.

        Respond in the following format:
        ENOUGH: YES/NO
        NEXT_QUERY: <your next query if NO, otherwise leave blank>
        """

        response = model.generate_content(
            planning_prompt
        )

        result = response.text

        if "ENOUGH: YES" in result:
            break

        try:
            current_query = result.split(
                "NEXT_QUERY:"
            )[1].strip()

        except:
            break

    final_context = "\n\n".join(memory)

    final_prompt = f"""
    Based on the following context, answer the question concisely and directly.

    Context:
    {final_context}

    Question:
    {query}

    Answer:
    """
    final_answer = model.generate_content(
        final_prompt
    )

    return final_answer.text

In [20]:
query = """ What is attention and
how does it improve
transformers? """

In [21]:
agentic_rag_answer(query, db)

'Attention is a mechanism that draws global dependencies between input and output.\n\nIt improves transformers by:\n*   Allowing the model to eschew recurrence, enabling significantly more parallelization.\n*   Reducing the number of operations required to relate signals from two arbitrary input or output positions to a constant.\n*   Counteracting reduced effective resolution through multi-head attention.\n*   Allowing every position in the decoder to attend over all positions in the input sequence.\n*   Helping to achieve a new state of the art in translation quality.\n*   Potentially yielding more interpretable models, as individual attention heads learn different tasks related to syntactic and semantic structure.'